In [ ]:
import numpy as np
from numpy.typing import NDArray
from itertools import product
from tqdm import tqdm
from math import factorial, comb

In [2]:
BOARD_LENGTH = 5
items = [0, 1]
allLayouts = product(items, repeat=BOARD_LENGTH ** 2)
result_dict: dict[int, tuple[int, int, int]] = {}
for i in range(22):
    result_dict[i] = (0, 0, 0)

def IsBingo(board: NDArray[np.bool]) -> bool:
    if BOARD_LENGTH in np.sum(board, axis=0):
        return True
    if BOARD_LENGTH in np.sum(board, axis=1):
        return True
    if np.sum(np.diag(board)) == BOARD_LENGTH:
        return True
    if np.sum(np.diag(np.fliplr(board))) == BOARD_LENGTH:
        return True
    return False

def GetNextIndex(currentIndex: tuple[int, int]) -> tuple[int, int]:
    if currentIndex[1] == BOARD_LENGTH - 1:
        return (currentIndex[0] + 1, 0)
    return (currentIndex[0], currentIndex[1] + 1)

def ValidBingoCheck(board: NDArray[np.bool]) -> int:
    """A Bingo is "valid" if there exists a tile such that removing the tile removes the bingo"""
    """Therefore, an "invalid" bingo has the following two properties:
        1. It must have a bingo
        2. There is no tile such that removing it would remove the Bingo
    Output: result enum:
    0 - Invalid
    1 - Bingo layout, valid, unique keystone
    2 - Bingo layout, valid, non-unique keystone
    3 - Not a bingo
    """
    if not IsBingo(board):
        return 3
    
    numberOfKeystones: int = 0
    currentIndex = (0,0)
    copy_board = board.copy()
    while currentIndex[0] < BOARD_LENGTH and currentIndex[1] < BOARD_LENGTH:
        if copy_board[currentIndex] == 1:
            copy_board[currentIndex] = 0
            if not IsBingo(copy_board):
                numberOfKeystones += 1
        nextIndex = GetNextIndex(currentIndex)
        currentIndex = nextIndex
        copy_board = board.copy() 

    if numberOfKeystones == 1:
        return 1
    elif numberOfKeystones > 1:
        return 2
    return 0

def Test():
    for layout in tqdm(allLayouts):
        numpified_layout = np.array(layout).reshape(BOARD_LENGTH,BOARD_LENGTH)
        result = ValidBingoCheck(numpified_layout)
        if result != 0:
            total_slots = np.sum(numpified_layout).astype(int)
            if result == 3: # Not a bingo
                result_dict[total_slots] = (result_dict[total_slots][0], result_dict[total_slots][1], result_dict[total_slots][2] + 1)
            elif result == 1: # Valid, bingo layout, unique keystone
                result_dict[total_slots] = (result_dict[total_slots][0] + 1, result_dict[total_slots][1], result_dict[total_slots][2])
            elif result == 2: # Valid, bingo layout, non-unique keystone
                result_dict[total_slots] = (result_dict[total_slots][0], result_dict[total_slots][1] + 1, result_dict[total_slots][2])
Test()
result_dict

33554432it [30:52, 18111.29it/s]


{0: (0, 0, 1),
 1: (0, 0, 25),
 2: (0, 0, 300),
 3: (0, 0, 2300),
 4: (0, 0, 12650),
 5: (0, 12, 53118),
 6: (0, 240, 176860),
 7: (0, 2280, 478420),
 8: (0, 13680, 1067895),
 9: (46, 58048, 1984881),
 10: (736, 184536, 3083468),
 11: (5520, 453480, 3998100),
 12: (25616, 874664, 4297872),
 13: (81544, 1328328, 3780704),
 14: (186096, 1575736, 2664548),
 15: (308388, 1430612, 1456572),
 16: (367120, 958328, 587935),
 17: (303041, 445644, 162362),
 18: (161872, 129888, 27088),
 19: (49332, 19788, 2176),
 20: (6776, 1064, 48),
 21: (240, 0, 0)}

In [6]:
p_z: dict[int, float] = {}

a_5, b_5, c_5 = result_dict[BOARD_LENGTH]
p_z[BOARD_LENGTH] = factorial(BOARD_LENGTH - 1) * float((a_5 + b_5 * BOARD_LENGTH) / (a_5 * factorial(BOARD_LENGTH - 1) + BOARD_LENGTH * b_5 * factorial(BOARD_LENGTH - 1) + c_5 * factorial(BOARD_LENGTH)))

for k in range(BOARD_LENGTH + 1, 22):
    a_k, b_k, c_k = result_dict[k]
    p_z[k] = factorial(k - 1) *((a_k + b_k * BOARD_LENGTH) / (a_k * factorial(k-1) + 5 * b_k * factorial(k-1) + c_k * factorial(k))) * (1 - np.sum([p_z[j] for j in range(BOARD_LENGTH, k)]))

E_Z = np.sum([k * p_z[k] for k in p_z.keys()])

print(np.sum([p_z[k] for k in p_z.keys()]))

E_Z

1.0


np.float64(14.897172583477603)

In [ ]:
def pmf(N: int, S: int, fails: int, successes: int) -> float:
    return comb(successes + fails - 1, fails) * comb(N-fails-successes, N-S-fails) / comb(N, N-S)

In [8]:
N = 75
S = 25
X_low = 5
X_high = 71
Z_low = 5
Z_high = 21

P_X_k = np.array([np.sum([p_z[j] * pmf(N=N, S=S, fails=k-j, successes=j) for j in range(max(Z_low, k - 50), min(k, Z_high) + 1)]) for k in range(X_low, X_high + 1)])

E_X = np.sum([k * P_X_k[i] for i,k in enumerate(range(X_low, X_high + 1))])
E_X

np.float64(43.54558139785762)